# Hands-on Task 1: Extend the Basic Pipeline (No LLM)
## Objective
Enhance an existing pipeline by adding one more processing step.
## Task
Introduce a new node called word_count with the following functionality:
•	Count the number of words in the input text
•	Store the result in a new state field:
word_count: int
Update the pipeline flow as follows:
START → count → word_count → summarise → END
Modify the make_summary node so that the final output includes both:
•	Character count
•	Word count
## Expected Output
Summary: 'LangGraph makes stateful AI workflows easy!' — 43 characters, 6 words.
## Skills Tested
•	Extending TypedDict state
•	Adding and connecting new nodes
•	Passing state across nodes
•	Using add_edge


In [9]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [10]:
#Create the state
class TextState(TypedDict):
    text: str
    char_count: int
    word_count: int
    summary: str

In [11]:
#Create character count node
def count_chars(state: TextState):

    return {
        "char_count": len(state["text"])
    }

In [12]:
#Create word count node
def count_words(state: TextState):

    return {
        "word_count": len(state["text"].split())
    }

In [13]:
#Create Summary Node
def make_summary(state: TextState):

    return {
        "summary":
        f"Summary: '{state['text']}' — "
        f"{state['char_count']} characters, "
        f"{state['word_count']} words."
    }

In [14]:
#Building the graph
builder = StateGraph(TextState)

builder.add_node("count", count_chars)
builder.add_node("word_count", count_words)
builder.add_node("summarise", make_summary)

builder.add_edge(START, "count")
builder.add_edge("count", "word_count")
builder.add_edge("word_count", "summarise")
builder.add_edge("summarise", END)

graph = builder.compile()


In [15]:
result = graph.invoke(
    {
        "text": "LangGraph makes stateful AI workflows easy!"
    }
)
print(result["summary"])

Summary: 'LangGraph makes stateful AI workflows easy!' — 43 characters, 6 words.


# Hands-on Task 2: Build a Review Pipeline with Claude
## Objective
Create a multi-step LLM pipeline with sequential processing.
##Task
Define a state structure:
class ReviewState(TypedDict):
    product: str
    review: str
    sentiment: str
    reply: str
``
## Build the following pipeline:
START → review_node → sentiment_node → reply_node → END
Node Requirements
1. review_node
•	Generate a short product review using Claude
2. sentiment_node
•	Classify the review into: 
o	Positive
o	Negative
o	Neutral
3. reply_node
•	Generate a one-line brand response based on the sentiment
## Test Input
product = "wireless noise-cancelling headphones"
## Skills Tested
•	Chaining multiple LLM steps
•	State management across nodes
•	Prompt design for generation and classification
•	Using Claude in graph-based workflows

In [16]:
from dotenv import dotenv_values
from langchain_openai import ChatOpenAI
config = dotenv_values("/home/ubuntu/Desktop/AgenticAI/.env")
KEY = config["KEY"]

llm = ChatOpenAI(
    api_key=KEY,
    base_url="https://llmgw-wp.tekstac.com/v1",
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0
)

In [17]:
class ReviewState(TypedDict):
    product: str
    review: str
    sentiment: str
    reply: str

In [18]:
def review_node(state):

    prompt = f"""
    Write a short product review for:
    {state['product']}
    """

    review = llm.invoke(prompt).content

    return {
        "review": review
    }

In [19]:
def sentiment_node(state):
    prompt = f"""
    Classify the following review as:

    Positive
    Negative
    Neutral

    Review:
    {state['review']}

    Return only one word.
    """

    sentiment = llm.invoke(prompt).content.strip()

    return {
        "sentiment": sentiment
    }

In [20]:
def reply_node(state):

    prompt = f"""
    Review:
    {state['review']}

    Sentiment:
    {state['sentiment']}

    Generate a one-line brand response.
    """

    reply = llm.invoke(prompt).content

    return {
        "reply": reply
    }

In [22]:
builder = StateGraph(ReviewState)

builder.add_node("review_node", review_node)
builder.add_node("sentiment_node", sentiment_node)
builder.add_node("reply_node", reply_node)

builder.add_edge(START, "review_node")
builder.add_edge("review_node", "sentiment_node")
builder.add_edge("sentiment_node", "reply_node")
builder.add_edge("reply_node", END)

review_graph = builder.compile()

In [24]:
result = review_graph.invoke(
    {
        "product": "wireless noise-cancelling headphones"
    }
)

print("Review:")
print(result["review"])

print("\nSentiment:")
print(result["sentiment"])

print("\nReply:")
print(result["reply"])

Review:
# Wireless Noise-Cancelling Headphones Review

**Rating: ★★★★★ (5/5)**

These headphones are a game-changer for anyone seeking peace and quiet. The noise-cancellation technology is remarkably effective—whether you're on a plane, in a busy office, or just want to escape the world, these deliver impressive silence.

**Pros:**
- Excellent active noise cancellation that actually works
- Comfortable for extended wear
- Great battery life (30+ hours)
- Crystal-clear sound quality across all frequencies
- Intuitive touch controls

**Cons:**
- Premium price point
- Takes a bit to find the perfect fit

**Bottom Line:**
If you value audio quality and tranquility, these headphones are worth the investment. They've transformed my commute and work-from-home experience. Highly recommended for audiophiles and casual listeners alike.

Sentiment:
Positive

Reply:
"Thank you for the glowing review! We're thrilled these headphones have enhanced your daily experience—your feedback truly inspires u

Hands-on Task 3: Three-Way Conditional Router
Objective
Implement a routing system with multiple decision branches.
Task
Create a router that classifies user questions into:
science
history
general
Based on this classification, route the input to one of three nodes:
science_node
history_node
general_node
``
Node Behavior
•	science_node → Respond as a science teacher
•	history_node → Respond as a historian
•	general_node → Respond as a general assistant
Routing Requirement
The routing function must return:
Literal["science_node", "history_node", "general_node"]
Testing Requirement
Test with at least 4 questions, ensuring all categories are covered.
Skills Tested
•	Using add_conditional_edges
•	Implementing multi-branch routing
•	Applying Literal type hints
•	Designing persona-based responses
________________________________________
Expected Outcome
By completing these tasks, you will be able to:
•	Extend pipeline state using TypedDict
•	Design multi-node workflows
•	Build LLM-powered pipelines
•	Route inputs dynamically across multiple branches
•	Apply persona-based response strategies


In [25]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

In [26]:
class QuestionState(TypedDict):
    question: str
    answer: str

In [27]:
def router(
    state: QuestionState
) -> Literal[
    "science_node",
    "history_node",
    "general_node"
]:

    question = state["question"].lower()

    if any(word in question for word in [
        "gravity",
        "atom",
        "planet",
        "science",
        "physics"
    ]):
        return "science_node"

    elif any(word in question for word in [
        "war",
        "history",
        "king",
        "empire",
        "independence"
    ]):
        return "history_node"

    else:
        return "general_node"

In [28]:
def science_node(state: QuestionState):

    return {
        "answer":
        f"Science Teacher: {state['question']} relates to scientific concepts and principles."
    }

In [29]:
def history_node(state: QuestionState):

    return {
        "answer":
        f"Historian: {state['question']} can be explained through historical events."
    }

In [30]:
def general_node(state: QuestionState):

    return {
        "answer":
        f"General Assistant: Here is a response to '{state['question']}'."
    }

In [31]:
builder = StateGraph(QuestionState)

builder.add_node("science_node", science_node)
builder.add_node("history_node", history_node)
builder.add_node("general_node", general_node)

builder.add_conditional_edges(
    START,
    router
)

builder.add_edge("science_node", END)
builder.add_edge("history_node", END)
builder.add_edge("general_node", END)

router_graph = builder.compile()

In [32]:
questions = [
    "What is gravity?",
    "How do atoms work?",
    "Who started World War 1?",
    "What is the capital of Japan?"
]

for question in questions:

    result = router_graph.invoke(
        {
            "question": question
        }
    )

    print("\nQuestion:", question)
    print("Answer:", result["answer"])


Question: What is gravity?
Answer: Science Teacher: What is gravity? relates to scientific concepts and principles.

Question: How do atoms work?
Answer: Science Teacher: How do atoms work? relates to scientific concepts and principles.

Question: Who started World War 1?
Answer: Historian: Who started World War 1? can be explained through historical events.

Question: What is the capital of Japan?
Answer: General Assistant: Here is a response to 'What is the capital of Japan?'.
